# 🔥 Remote Work Burnout Prediction
**Dataset:** `work_from_home_burnout_dataset.csv`  
**Goals:**
- Classify employees into burnout risk categories (Low / High)
- Predict burnout score (regression)
- Identify which work habits drive burnout most

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold

# Imbalance handling
from imblearn.over_sampling import SMOTE

# Models
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

# Evaluation
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, confusion_matrix, accuracy_score,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

print('All libraries imported successfully.')

## 2. Load & Inspect Data

In [ ]:
df = pd.read_csv('work_from_home_burnout_dataset.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print()
print('Class distribution:')
print(df['burnout_risk'].value_counts())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# --- Distribution plots for key features split by burnout risk ---
key_features = ['work_hours', 'sleep_hours', 'screen_time_hours', 'breaks_taken',
                'meetings_count', 'task_completion_rate']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    for risk, color in zip(['Low', 'High'], ['steelblue', 'tomato']):
        subset = df[df['burnout_risk'] == risk][feat]
        axes[i].hist(subset, bins=20, alpha=0.6, color=color, label=risk)
    axes[i].set_title(f'{feat} by Burnout Risk')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Count')
    axes[i].legend()

plt.suptitle('Feature Distributions by Burnout Risk', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Class distribution ---
plt.figure(figsize=(5, 4))
counts = df['burnout_risk'].value_counts()
colors = ['steelblue', 'tomato']
plt.bar(counts.index, counts.values, color=colors)
for i, v in enumerate(counts.values):
    plt.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.title('Burnout Risk Class Distribution')
plt.ylabel('Count')
plt.show()
print('Note: Severe class imbalance — High risk is a tiny minority.')

In [ ]:
# --- Scatter plots ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for risk, color in zip(['Low', 'High'], ['steelblue', 'tomato']):
    subset = df[df['burnout_risk'] == risk]
    axes[0].scatter(subset['work_hours'], subset['burnout_score'],
                    alpha=0.4, color=color, label=risk)
    axes[1].scatter(subset['sleep_hours'], subset['burnout_score'],
                    alpha=0.4, color=color, label=risk)

axes[0].set(title='Work Hours vs Burnout Score', xlabel='Work Hours', ylabel='Burnout Score')
axes[0].legend()
axes[1].set(title='Sleep Hours vs Burnout Score', xlabel='Sleep Hours', ylabel='Burnout Score')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Correlation heatmap ---
plt.figure(figsize=(10, 6))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, cbar_kws={'label': 'Correlation'})
plt.title('Feature Correlation Heatmap', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Box plots to see spread by burnout risk ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, feat in zip(axes, ['work_hours', 'sleep_hours', 'burnout_score']):
    df.boxplot(column=feat, by='burnout_risk', ax=ax,
               boxprops=dict(color='steelblue'),
               medianprops=dict(color='tomato', linewidth=2))
    ax.set_title(f'{feat} by Risk')
    ax.set_xlabel('Burnout Risk')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 4. Preprocessing

In [ ]:
# Encode day_type
le = LabelEncoder()
df['day_type_enc'] = le.fit_transform(df['day_type'])

features = [
    'day_type_enc', 'work_hours', 'screen_time_hours',
    'meetings_count', 'breaks_taken', 'after_hours_work',
    'sleep_hours', 'task_completion_rate'
]
feature_labels = [
    'Day Type', 'Work Hours', 'Screen Time',
    'Meetings Count', 'Breaks Taken', 'After-Hours Work',
    'Sleep Hours', 'Task Completion Rate'
]

X = df[features]

# Binary target: Low=0, High=1  (map Medium->Low for binary classification)
y_class = df['burnout_risk'].map({'Low': 0, 'Medium': 0, 'High': 1})

# Regression target
y_reg = df['burnout_score']

print('Feature matrix shape:', X.shape)
print('Class distribution after mapping:')
print(y_class.value_counts())

## 5. Train / Test Split + SMOTE

In [ ]:
# Split
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class
)

# Apply SMOTE on training set only
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_c, y_train_c)

# Scale
scaler_c = StandardScaler()
X_train_smote_scaled = scaler_c.fit_transform(X_train_smote)
X_test_c_scaled = scaler_c.transform(X_test_c)

# Show class balance after SMOTE
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, data, title in zip(axes,
                            [y_train_c, y_train_smote],
                            ['Before SMOTE', 'After SMOTE']):
    counts = data.value_counts().sort_index()
    ax.bar(['Low (0)', 'High (1)'], counts.values, color=['steelblue', 'tomato'])
    for i, v in enumerate(counts.values):
        ax.text(i, v + 2, str(v), ha='center', fontweight='bold')
    ax.set_title(title)
    ax.set_ylabel('Count')
plt.suptitle('Class Distribution Before vs After SMOTE', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Model Training

In [ ]:
# ---- Random Forest ----
rf_clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf_clf.fit(X_train_smote_scaled, y_train_smote)
rf_preds = rf_clf.predict(X_test_c_scaled)
rf_proba = rf_clf.predict_proba(X_test_c_scaled)[:, 1]
print('Random Forest trained.')

In [ ]:
# ---- Logistic Regression ----
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
log_reg.fit(X_train_smote_scaled, y_train_smote)
lr_preds = log_reg.predict(X_test_c_scaled)
lr_proba = log_reg.predict_proba(X_test_c_scaled)[:, 1]
print('Logistic Regression trained.')

In [ ]:
# ---- Gradient Boosting (NEW) ----
gb_clf = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                     max_depth=4, random_state=42)
gb_clf.fit(X_train_smote_scaled, y_train_smote)
gb_preds = gb_clf.predict(X_test_c_scaled)
gb_proba = gb_clf.predict_proba(X_test_c_scaled)[:, 1]
print('Gradient Boosting trained.')

## 7. Threshold Tuning (Fix for Zero High-Risk Detection)

In [ ]:
# --- Find best threshold for RF using F1 score on High class ---
from sklearn.metrics import f1_score

thresholds = np.arange(0.10, 0.90, 0.05)
f1_scores = []

for t in thresholds:
    preds_t = (rf_proba >= t).astype(int)
    f1 = f1_score(y_test_c, preds_t, pos_label=1, zero_division=0)
    f1_scores.append(f1)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

plt.figure(figsize=(8, 4))
plt.plot(thresholds, f1_scores, marker='o', color='steelblue', linewidth=2)
plt.axvline(best_threshold, color='tomato', linestyle='--',
            label=f'Best threshold = {best_threshold:.2f}')
plt.title('F1 Score (High class) vs Decision Threshold — Random Forest')
plt.xlabel('Threshold')
plt.ylabel('F1 Score (High class)')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Best threshold: {best_threshold:.2f}  |  F1 at best: {f1_scores[best_idx]:.3f}')

In [ ]:
# Apply tuned threshold to RF predictions
rf_preds_tuned = (rf_proba >= best_threshold).astype(int)

print(f'=== Random Forest — Default threshold (0.50) ===')
print(classification_report(y_test_c, rf_preds, target_names=['Low', 'High']))

print(f'=== Random Forest — Tuned threshold ({best_threshold:.2f}) ===')
print(classification_report(y_test_c, rf_preds_tuned, target_names=['Low', 'High']))

## 8. Model Evaluation — Classification Reports

In [ ]:
models = {
    'Random Forest (tuned)': (rf_preds_tuned, rf_proba),
    'Logistic Regression':   (lr_preds,        lr_proba),
    'Gradient Boosting':     (gb_preds,        gb_proba),
}

for name, (preds, proba) in models.items():
    print(f'====== {name} ======')
    print(classification_report(y_test_c, preds, target_names=['Low', 'High']))
    print()

## 9. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
cmaps = ['Blues', 'Greens', 'Oranges']

for ax, (name, (preds, _)), cmap in zip(axes, models.items(), cmaps):
    disp = ConfusionMatrixDisplay(
        confusion_matrix(y_test_c, preds),
        display_labels=['Low', 'High']
    )
    disp.plot(ax=ax, colorbar=False, cmap=cmap)
    ax.set_title(name, fontsize=11)

plt.suptitle('Confusion Matrices — All Models', fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

## 10. ROC-AUC Curves

In [ ]:
plt.figure(figsize=(7, 5))
colors = ['steelblue', 'seagreen', 'darkorange']

for (name, (preds, proba)), color in zip(models.items(), colors):
    fpr, tpr, _ = roc_curve(y_test_c, proba)
    auc = roc_auc_score(y_test_c, proba)
    plt.plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Burnout Risk Classification', fontsize=13)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 11. Cross-Validation (5-Fold)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_all_scaled = scaler_c.fit_transform(X)  # scale full dataset for CV

cv_results = {}
cv_models = {
    'Random Forest':     RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'),
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                    max_depth=4, random_state=42),
}

for name, model in cv_models.items():
    scores = cross_val_score(model, X_all_scaled, y_class, cv=cv,
                             scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:25s}  AUC: {scores.mean():.3f} ± {scores.std():.3f}')

In [ ]:
# Visualise CV results
fig, ax = plt.subplots(figsize=(8, 4))
names = list(cv_results.keys())
means = [cv_results[n].mean() for n in names]
stds  = [cv_results[n].std()  for n in names]
colors = ['steelblue', 'seagreen', 'darkorange']

bars = ax.bar(names, means, yerr=stds, capsize=6,
              color=colors, alpha=0.8, error_kw={'linewidth': 2})
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{mean:.3f}', ha='center', fontweight='bold', fontsize=10)
ax.set_ylim(0, 1.1)
ax.set_ylabel('ROC-AUC (5-fold CV)')
ax.set_title('Cross-Validation ROC-AUC Scores (mean ± std)', fontsize=13)
plt.tight_layout()
plt.show()

## 12. Model Comparison Table

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

comparison_rows = []
for name, (preds, proba) in models.items():
    comparison_rows.append({
        'Model': name,
        'Accuracy':  round(accuracy_score(y_test_c, preds), 3),
        'Precision (High)': round(precision_score(y_test_c, preds, pos_label=1, zero_division=0), 3),
        'Recall (High)':    round(recall_score(y_test_c, preds, pos_label=1, zero_division=0), 3),
        'F1 (High)':        round(f1_score(y_test_c, preds, pos_label=1, zero_division=0), 3),
        'ROC-AUC':  round(roc_auc_score(y_test_c, proba), 3),
    })

comparison_df = pd.DataFrame(comparison_rows).set_index('Model')
print('=== Model Comparison ===')
comparison_df

## 13. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Random Forest importance
rf_imp = pd.Series(rf_clf.feature_importances_, index=feature_labels).sort_values()
colors_rf = ['tomato' if v == rf_imp.max() else 'steelblue' for v in rf_imp.values]
rf_imp.plot(kind='barh', ax=axes[0], color=colors_rf)
axes[0].set_title('Feature Importance — Random Forest', fontsize=12)
axes[0].set_xlabel('Importance Score')
axes[0].axvline(rf_imp.mean(), color='gray', linestyle='--', alpha=0.7, label='Mean')
axes[0].legend()

# Gradient Boosting importance
gb_imp = pd.Series(gb_clf.feature_importances_, index=feature_labels).sort_values()
colors_gb = ['tomato' if v == gb_imp.max() else 'darkorange' for v in gb_imp.values]
gb_imp.plot(kind='barh', ax=axes[1], color=colors_gb)
axes[1].set_title('Feature Importance — Gradient Boosting', fontsize=12)
axes[1].set_xlabel('Importance Score')
axes[1].axvline(gb_imp.mean(), color='gray', linestyle='--', alpha=0.7, label='Mean')
axes[1].legend()

plt.suptitle('Which Features Drive Burnout Most?', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 14. Regression — Predict Burnout Score

In [ ]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled  = scaler_r.transform(X_test_r)

rf_reg = RandomForestRegressor(n_estimators=200, random_state=42)
rf_reg.fit(X_train_r_scaled, y_train_r)
reg_preds = rf_reg.predict(X_test_r_scaled)

mae  = mean_absolute_error(y_test_r, reg_preds)
rmse = np.sqrt(mean_squared_error(y_test_r, reg_preds))
r2   = r2_score(y_test_r, reg_preds)

print(f'Regression Results:')
print(f'  MAE  : {mae:.2f}')
print(f'  RMSE : {rmse:.2f}')
print(f'  R²   : {r2:.3f}')

In [ ]:
# Predicted vs Actual scatter + residual plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Predicted vs Actual
axes[0].scatter(y_test_r, reg_preds, alpha=0.4, color='steelblue', edgecolors='white', s=30)
min_v, max_v = y_test_r.min(), y_test_r.max()
axes[0].plot([min_v, max_v], [min_v, max_v], 'r--', lw=1.5, label='Perfect prediction')
axes[0].set(title='Predicted vs Actual Burnout Score',
            xlabel='Actual Burnout Score', ylabel='Predicted Burnout Score')
axes[0].legend()
axes[0].text(0.05, 0.92, f'R² = {r2:.3f}', transform=axes[0].transAxes,
             fontsize=11, color='steelblue')

# Residuals
residuals = y_test_r - reg_preds
axes[1].scatter(reg_preds, residuals, alpha=0.4, color='darkorange', edgecolors='white', s=30)
axes[1].axhline(0, color='red', linestyle='--', lw=1.5)
axes[1].set(title='Residuals vs Predicted',
            xlabel='Predicted Burnout Score', ylabel='Residual (Actual − Predicted)')

plt.suptitle('Regression Model — Random Forest', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 15. Predict for a New Employee Sample

In [ ]:
# Sample employee profile
sample = pd.DataFrame({
    'day_type_enc':         [1],      # Weekday
    'work_hours':           [9.5],
    'screen_time_hours':    [12.2],
    'meetings_count':       [5],
    'breaks_taken':         [1],
    'after_hours_work':     [1],
    'sleep_hours':          [5.6],
    'task_completion_rate': [60]
})

sample_scaled_c = scaler_c.transform(sample)
sample_scaled_r = scaler_r.transform(sample)

# Classification predictions
rf_label  = 'High' if rf_clf.predict_proba(sample_scaled_c)[0, 1] >= best_threshold else 'Low'
lr_label  = 'High' if log_reg.predict_proba(sample_scaled_c)[0, 1] >= 0.5           else 'Low'
gb_label  = 'High' if gb_clf.predict_proba(sample_scaled_c)[0, 1] >= 0.5            else 'Low'

# Probabilities
rf_prob   = rf_clf.predict_proba(sample_scaled_c)[0, 1]
lr_prob   = log_reg.predict_proba(sample_scaled_c)[0, 1]
gb_prob   = gb_clf.predict_proba(sample_scaled_c)[0, 1]

# Regression
predicted_score = rf_reg.predict(sample_scaled_r)[0]

print('=== Prediction for Sample Employee ===')
print(f'Random Forest     : {rf_label}  (P(High) = {rf_prob:.3f})')
print(f'Logistic Regression: {lr_label}  (P(High) = {lr_prob:.3f})')
print(f'Gradient Boosting : {gb_label}  (P(High) = {gb_prob:.3f})')
print(f'Predicted Burnout Score: {predicted_score:.1f}')

## 16. Summary & Key Findings

In [ ]:
print('=' * 55)
print('           PROJECT SUMMARY')
print('=' * 55)
print()
print('DATASET')
print(f'  Rows      : {df.shape[0]}')
print(f'  Features  : {len(features)}')
print(f'  High-risk : {y_class.sum()} / {len(y_class)} ({y_class.mean()*100:.1f}%)')
print()
print('BEST MODEL  : ' + comparison_df['ROC-AUC'].idxmax())
print(f'  ROC-AUC   : {comparison_df["ROC-AUC"].max():.3f}')
print(f'  F1 (High) : {comparison_df["F1 (High)"].max():.3f}')
print()
print('TOP BURNOUT DRIVERS (Random Forest)')
for feat, imp in rf_imp.sort_values(ascending=False).head(3).items():
    print(f'  {feat:25s}: {imp:.3f}')
print()
print('REGRESSION (Burnout Score)')
print(f'  R²   : {r2:.3f}')
print(f'  MAE  : {mae:.2f}')
print(f'  RMSE : {rmse:.2f}')
print('=' * 55)